In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2

print(os.getcwd())

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/2_Regional_analaysis


In [36]:
import pandas as pd
import pickle
import numpy as np

In [7]:

data = []

for notebook in range(1,5):
    
    with open(f"{notebook}_results.pkl", "rb") as f:
        loaded_data = pickle.load(f)
        data.append(loaded_data)

data = pd.concat(data, ignore_index=True)


In [9]:
print(data.head())

        Dataset Training sub-population Metric type  Seed  \
0  Hypertension             London, P=0         Ctd     1   
1  Hypertension             London, P=0         Ctd     1   
2  Hypertension             London, P=0         IBS     1   
3  Hypertension             London, P=0         IBS     1   
4  Hypertension             London, P=0       INBLL     1   

  Evaluation sub-population  Metric value  
0                    London      0.827226  
1                North-East      0.839028  
2                    London      0.074218  
3                North-East      0.069355  
4                    London      0.237391  


# Results

In [26]:
def get_table(dataframe, by_metric="Ctd"):

    data_metric = dataframe[dataframe["Metric type"] == by_metric]

    # group by training/evaluation sub-pop and aggregate
    summary = (
        data_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Metric value"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(3).astype(str) #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [27]:
table_ctd = get_table(data, by_metric="Ctd")
print(table_ctd)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                0.827      0.839
London, P=1                 0.84      0.849
North East, P=0            0.754      0.748
North East, P=1            0.817      0.836


In [28]:
table_ibs = get_table(data, by_metric="IBS")
print(table_ibs)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                0.074      0.069
London, P=1                0.072      0.067
North East, P=0            0.081      0.078
North East, P=1            0.077      0.071


In [29]:
table_inbll = get_table(data, by_metric="INBLL")
print(table_inbll)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                0.237      0.223
London, P=1                0.231      0.215
North East, P=0            0.263      0.256
North East, P=1             0.25      0.232


# Results (scaled)

We now scale the results to see relative differences. Using the equation 

$$\frac{Eval(Y \mid P, X)}{Eval(X \mid P=0, X)}$$

where Y is the evaluation population, trained on sub-population X, with P a binary indicator denoting whether we intiialised from a pre-trained model (1), or not (0).

In [96]:
def get_scaled_table(df, by_metric="Ctd"):

    # Get the dataset type
    dataset = df["Dataset"][0]
    
    df_metric = df.loc[df["Metric type"] == by_metric].copy()
    df_metric["Scaled metric change"] = np.nan

    for idx, row in df_metric.iterrows():
        seed = row["Seed"]
        training_full = row["Training sub-population"]
        training_base = training_full.split(",")[0]
        row_value = row["Metric value"]

        # scaler
        scaler_row = df_metric[
            (df_metric["Seed"] == seed) &
            (df_metric["Training sub-population"] == training_base + ", P=0") &
            (df_metric["Evaluation sub-population"] == training_base.replace(" ", "-"))
        ]
        scaler = scaler_row["Metric value"].iloc[0]

        df_metric.at[idx, "Scaled metric change"] = ((row_value / scaler) - 1) * 100

    # group by training/evaluation sub-pop and aggregate
    summary = (
        df_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Scaled metric change"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(2).astype(str) + "%" #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [97]:
scaled_table_ctd = get_scaled_table(data, by_metric="Ctd")
print(scaled_table_ctd)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                 0.0%      1.43%
London, P=1                1.57%      2.58%
North East, P=0            0.71%       0.0%
North East, P=1            9.15%     11.69%


In [98]:
scaled_table_ibs = get_scaled_table(data, by_metric="IBS")
print(scaled_table_ibs)

Evaluation sub-population  London North-East
Training sub-population                     
London, P=0                  0.0%     -6.55%
London, P=1                -2.82%    -10.24%
North East, P=0             4.41%       0.0%
North East, P=1            -0.65%     -8.68%


In [99]:
scaled_table_inbll = get_scaled_table(data, by_metric="INBLL")
print(scaled_table_inbll)

Evaluation sub-population  London North-East
Training sub-population                     
London, P=0                  0.0%     -6.02%
London, P=1                -2.67%     -9.43%
North East, P=0             2.89%       0.0%
North East, P=1            -2.23%      -9.5%


# Results (scaled)

We now scale the results to see relative differences, but do not assume an model that was not pre-trained as the baseline. Using the equation 

$$\frac{Eval(Y \mid P, X)}{Eval(X \mid P, X)}$$

where Y is the evaluation population, trained on sub-population X, with P a binary indicator denoting whether we intiialised from a pre-trained model (1), or not (0).

In [101]:
def get_scaled2_table(df, by_metric="Ctd"):

    # Get the dataset type
    dataset = df["Dataset"][0]
    
    df_metric = df.loc[df["Metric type"] == by_metric].copy()
    df_metric["Scaled metric change"] = np.nan

    for idx, row in df_metric.iterrows():
        seed = row["Seed"]
        training_full = row["Training sub-population"]
        training_base = training_full.split(",")[0]
        row_value = row["Metric value"]

        # scaler
        scaler_row = df_metric[
            (df_metric["Seed"] == seed) &
            (df_metric["Training sub-population"] == training_full) &
            (df_metric["Evaluation sub-population"] == training_base.replace(" ", "-"))
        ]
        scaler = scaler_row["Metric value"].iloc[0]

        df_metric.at[idx, "Scaled metric change"] = ((row_value / scaler) - 1) * 100

    # group by training/evaluation sub-pop and aggregate
    summary = (
        df_metric
        .groupby(["Training sub-population", "Evaluation sub-population"])["Scaled metric change"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    
    # Standard error
    summary["sem"] = summary["std"] / summary["count"]**0.5
    
    # Format the table contents (removed error bars until the results are obtained)
    summary["formatted"] = summary["mean"].round(2).astype(str) + "%" #+ " ± " + summary["std"].round(3).astype(str)
    
    # pivot into desired table
    table = summary.pivot(
        index="Training sub-population",
        columns="Evaluation sub-population",
        values="formatted"
    )
    return table

In [102]:
scaled_table_ctd = get_scaled2_table(data, by_metric="Ctd")
print(scaled_table_ctd)

Evaluation sub-population  London North-East
Training sub-population                     
London, P=0                  0.0%      1.43%
London, P=1                  0.0%       1.0%
North East, P=0             0.71%       0.0%
North East, P=1            -2.28%       0.0%


In [103]:
scaled_table_ibs = get_scaled2_table(data, by_metric="IBS")
print(scaled_table_ibs)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                 0.0%     -6.55%
London, P=1                 0.0%     -7.64%
North East, P=0            4.41%       0.0%
North East, P=1            8.79%       0.0%


In [104]:
scaled_table_inbll = get_scaled2_table(data, by_metric="INBLL")
print(scaled_table_inbll)

Evaluation sub-population London North-East
Training sub-population                    
London, P=0                 0.0%     -6.02%
London, P=1                 0.0%     -6.95%
North East, P=0            2.89%       0.0%
North East, P=1            8.04%       0.0%
